# 10 — Profiling, Memory, and Honest Benchmarking

Learn to measure where time and memory go before optimizing. Distinguish algorithmic complexity from wall-clock measurements and apply cProfile, timeit, and tracemalloc correctly.

*Advanced · 90–150 min*

## You will learn

- Collect timing measurements with timeit
- Interpret cProfile cumulative time and call counts
- Inspect Python allocation snapshots with tracemalloc
- Identify benchmark noise and non-equivalent workloads

## Examples

### 1. Big-O and practical cost

Algorithmic complexity describes growth, not absolute runtime. A set lookup is usually O(1) on average after O(n) construction, while a list lookup is O(n) in the worst case. For tiny inputs, a list may be simpler and faster overall.

In [ ]:
values = list(range(1000))
lookup_set = set(values)
queries = [0, 500, 999, 1200]
assert [x in values for x in queries] == [x in lookup_set for x in queries]
print([x in lookup_set for x in queries])

**Design review:** When would construction cost outweigh savings from set membership?

### 2. Repeatable timing

timeit measures repeated calls, reducing timer overhead and some noise. Compare equivalent behavior and include setup time where it matters. Results will vary with CPU load, Python implementation and input data.

In [ ]:
import timeit

values = list(range(5000))
lookup_set = set(values)
list_seconds = timeit.timeit(lambda: 4999 in values, number=150)
set_seconds = timeit.timeit(lambda: 4999 in lookup_set, number=150)
print("list:", round(list_seconds, 6), "set:", round(set_seconds, 6))

**Design review:** How would a query near index zero change the comparison?

### 3. Profiling call stacks

cProfile collects function call statistics and cumulative time. Profile representative workloads rather than a single trivial function. Profiling adds overhead, so use it to identify hotspots rather than as a precise production latency measurement.

In [ ]:
import cProfile, pstats
from io import StringIO

def workload():
    return sum(n * n for n in range(10000))

profiler = cProfile.Profile()
profiler.runcall(workload)
report = StringIO()
pstats.Stats(profiler, stream=report).sort_stats("cumulative").print_stats(6)
print(report.getvalue()[:650])

**Design review:** Why is 'cumulative' time different from time spent only in the function itself?

### 4. Tracking Python allocations

tracemalloc tracks memory allocations managed by Python and reports current/peak traced memory. It is not a full operating-system resident-memory profiler. Start tracing before the allocations you want to measure, then stop tracing afterward.

In [ ]:
import tracemalloc

tracemalloc.start()
sample = [str(i) for i in range(2000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print("count:", len(sample), "peak traced bytes:", peak)
assert peak >= current >= 0

**Design review:** Why can traced memory differ from the process's total memory use?

## Exercises

**Challenge 1.** Benchmark list, set, and dict lookup under a repeated-query workload.

In [ ]:
# Challenge 1
# TODO: Write implementation and assertions.

**Challenge 2.** Profile a data transformation and identify its dominant function.

In [ ]:
# Challenge 2
# TODO: Write implementation and assertions.

**Challenge 3.** Compare traced allocation peaks for list materialization vs generator iteration.

In [ ]:
# Challenge 3
# TODO: Write implementation and assertions.

**Challenge 4.** Write a benchmark methodology explaining warmups, sample size, and confidence limits.

In [ ]:
# Challenge 4
# TODO: Write implementation and assertions.

- One timing is not proof of a universal speedup.
- tracemalloc is not a complete RSS memory profiler.
- Measure before optimizing and test correctness after changes.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** timeit repeats measurements to…

   - **a)** Profile functions
   - **b)** Reduce noise
   - **c)** Use more memory

**2.** cProfile's cumulative time includes…

   - **a)** Time spent in functions it calls too
   - **b)** Only the function's own code
   - **c)** Import time only

**3.** tracemalloc measures…

   - **a)** Disk usage
   - **b)** CPU time
   - **c)** Python memory allocations

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/10', {1: '8f1194861272', 2: 'd5ed86c195fc', 3: '8392db161c11'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [11 — Algorithms, Complexity, and Cache Design](11_algorithms_complexity_caching.ipynb)